# 02 - Análisis intermedio: series de tiempo, países, NCM

Sigue de `01_introduccion.ipynb`. Acá:

- Series de tiempo mensuales y crecimiento interanual (funciones de ventana).
- Decodificar códigos (país, aduana, unidad, medio de transporte) con las
  tablas de `codigos/codigos_arca.py`.
- Rankings por posición NCM y por importador dentro de una NCM.
- Evitar duplicar `FOB_TOTAL_USD` al agregar (una fila por concepto de
  arancel, no una fila por ítem).

In [ ]:
import sys
import duckdb
import pandas as pd

sys.path.insert(0, "../codigos")
from codigos_arca import PAISES, ADUANAS, UNIDADES, MEDIOS_TRANSPORTE

HISTORICO = "../Data/impo_historico.parquet"
con = duckdb.connect()
con.execute("PRAGMA memory_limit='1.5GB'")  # ver notebook 03: por que esto importa en una maquina chica
con.execute("PRAGMA temp_directory='../Data'")
con.execute(f"CREATE VIEW impo AS SELECT * FROM read_parquet('{HISTORICO}')")


## Por qué agrupar antes de sumar FOB

La tabla tiene una fila por *concepto de arancel*, no una fila por ítem: un
mismo ítem puede tener varias líneas de tributo (derechos, tasa estadística,
IVA...) y todas repiten el mismo `FOB_UNITARIO_USD`/`FOB_TOTAL_USD`. Sumar esa
columna directo sobre `impo` sobrecuenta el FOB tantas veces como conceptos de
arancel tenga cada ítem. Agrupar primero por declaración+ítem con
`any_value` (que toma cualquier valor del grupo, total consistentes porque ya
vienen repetidos) antes de sumar.

In [ ]:
con.execute('''
    WITH items AS (
        SELECT PERIODO, DESTINACION, NUM_ITEM,
               any_value(FOB_UNITARIO_USD) AS fob_unitario,
               any_value(POS_NCM) AS ncm
        FROM impo
        GROUP BY PERIODO, DESTINACION, NUM_ITEM
    )
    SELECT substr(PERIODO, 1, 4) AS anio, round(sum(fob_unitario)) AS fob_usd
    FROM items
    GROUP BY 1 ORDER BY 1
''').fetchdf()


## Serie mensual y crecimiento interanual (YoY)

`lag(x, 12)` sobre una serie ordenada por mes trae el mismo mes del año anterior; de ahí sale el crecimiento interanual sin tener que hacer un self-join.

In [ ]:
serie = con.execute('''
    WITH items AS (
        SELECT PERIODO, DESTINACION, NUM_ITEM,
               any_value(FOB_UNITARIO_USD) AS fob_unitario
        FROM impo
        GROUP BY PERIODO, DESTINACION, NUM_ITEM
    ),
    mensual AS (
        SELECT PERIODO, sum(fob_unitario) AS fob_usd
        FROM items GROUP BY PERIODO
    )
    SELECT PERIODO,
           fob_usd,
           lag(fob_usd, 12) OVER (ORDER BY PERIODO) AS fob_usd_ano_anterior,
           round(100.0 * (fob_usd / lag(fob_usd, 12) OVER (ORDER BY PERIODO) - 1), 1) AS var_interanual_pct
    FROM mensual
    ORDER BY PERIODO
''').fetchdf()
serie.tail(12)


## Graficar la serie

In [ ]:
import matplotlib.pyplot as plt

serie["fecha"] = pd.to_datetime(serie["PERIODO"], format="%Y%m")
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(serie["fecha"], serie["fob_usd"])
ax.set_title("FOB total de importaciones por mes (USD)")
ax.set_xlabel("")
plt.tight_layout()


## Decodificar país de origen

`PAIS_ORIGEN` es un código; `codigos_arca.PAISES` lo traduce. Se registra como tabla auxiliar y se hace `LEFT JOIN` (por si aparece algún código no mapeado, no se pierde la fila).

In [ ]:
paises_df = pd.DataFrame(list(PAISES.items()), columns=["cod", "pais"])
con.register("t_pais", paises_df)

con.execute('''
    SELECT coalesce(p.pais, i.PAIS_ORIGEN) AS pais_origen,
           count(*) AS filas
    FROM impo i
    LEFT JOIN t_pais p ON p.cod = i.PAIS_ORIGEN
    WHERE i.PERIODO = (SELECT max(PERIODO) FROM impo)
    GROUP BY 1
    ORDER BY filas DESC
    LIMIT 15
''').fetchdf()


## Ranking de NCM por FOB en un año dado

`PERIODO BETWEEN '{ANIO}01' AND '{ANIO}12'` filtra por rango directo sobre
`PERIODO`, no `substr(PERIODO, 1, 4) = '{ANIO}'`: la primera forma deja que
DuckDB pode row groups enteros por las estadísticas min/max de Parquet (ver
notebook 03); envolver la columna en `substr` se lo esconde y fuerza un
escaneo completo del archivo.

In [ ]:
ANIO = "2024"
con.execute(f'''
    WITH items AS (
        SELECT DESTINACION, NUM_ITEM,
               any_value(POS_NCM) AS ncm,
               any_value(FOB_UNITARIO_USD) AS fob_unitario
        FROM impo
        WHERE PERIODO BETWEEN '{ANIO}01' AND '{ANIO}12'
        GROUP BY DESTINACION, NUM_ITEM
    )
    SELECT ncm, round(sum(fob_unitario)) AS fob_usd, count(*) AS items
    FROM items
    GROUP BY ncm
    ORDER BY fob_usd DESC
    LIMIT 20
''').fetchdf()


## Importadores dentro de una NCM puntual

In [ ]:
NCM_PREFIJO = "8504"
con.execute(f'''
    WITH items AS (
        SELECT DESTINACION, NUM_ITEM,
               any_value(NOMBRE_IMPORTADOR) AS importador,
               any_value(FOB_UNITARIO_USD) AS fob_unitario
        FROM impo
        WHERE POS_NCM LIKE '{NCM_PREFIJO}%'
        GROUP BY DESTINACION, NUM_ITEM
    )
    SELECT importador, round(sum(fob_unitario)) AS fob_usd, count(*) AS items
    FROM items
    GROUP BY importador
    ORDER BY fob_usd DESC
    LIMIT 20
''').fetchdf()


## Siguiente paso

`03_avanzado.ipynb`: por qué el orden del archivo importa para el
rendimiento, patrones de memoria acotada con polars/DuckDB, detección de
outliers en precios unitarios, y exportar subconjuntos grandes.